In [ ]:
# Kwaliteit classificatie (alleen opdrachtkern)
# Dit notebook bevat alleen de noodzakelijke stappen voor de opdracht.

In [13]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [15]:
def find_ames_file(filename='AmesHousing.xlsx'):
    explicit = Path(r'c:\Users\omarm\Desktop\opdrachty\AmesHousing.xlsx')
    direct = [explicit, Path(filename), Path.cwd() / filename, Path.cwd().parent / filename]
    for candidate in direct:
        if candidate.exists():
            return candidate

    found = list(Path.cwd().rglob(filename))
    if found:
        return found[0]

    looked_in = '\n'.join(str(p) for p in direct)
    raise FileNotFoundError(
        'AmesHousing.xlsx niet gevonden. Gezocht in:\n' + looked_in +
        '\nPlaats het bestand in c:\\Users\\omarm\\Desktop\\opdrachty.'
    )

ames_file = find_ames_file()
df = pd.read_excel(ames_file, sheet_name='AmesHousing')
print(f'Bestand geladen: {ames_file}')
df.head()

Bestand geladen: c:\Users\omarm\Desktop\opdrachty\AmesHousing.xlsx


,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story


## Data inlezen
Hieronder wordt het tabblad AmesHousing ingelezen als DataFrame.

In [16]:
target_col = 'Overall Qual'
print('Target:', target_col)
print('Rijen met target:', df[target_col].notna().sum())

Target: Overall Qual
Rijen met target: 2930


In [17]:
def run_quality_experiment(df, target_col, features, model_params, test_size=0.20):
    X = pd.get_dummies(df[features], drop_first=True).fillna(0)
    y = df[target_col]

    valid = y.notna()
    X = X.loc[valid]
    y = y.loc[valid]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    model = DecisionTreeClassifier(random_state=42, **model_params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }
    return metrics

In [18]:
experiments = [
    {
        'name': 'Exp 1',
        'features': ['Neighborhood', 'Gr Liv Area', 'Year Built'],
        'params': {'max_depth': 6, 'min_samples_leaf': 8},
    },
    {
        'name': 'Exp 2',
        'features': ['Neighborhood', 'Gr Liv Area', 'Year Built', 'Total Bsmt SF'],
        'params': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 12},
    },
    {
        'name': 'Exp 3',
        'features': ['Neighborhood', 'Gr Liv Area', 'Year Built', 'Lot Area', 'Full Bath'],
        'params': {'max_depth': 12, 'min_samples_leaf': 3, 'min_samples_split': 8},
    },
    {
        'name': 'Exp 4',
        'features': ['Neighborhood', 'House Style', 'Gr Liv Area', 'Year Built', 'Total Bsmt SF', 'Garage Cars'],
        'params': {'max_depth': 14, 'min_samples_leaf': 2, 'min_samples_split': 6},
    },
]

In [19]:
results = []
for exp in experiments:
    available = [c for c in exp['features'] if c in df.columns]
    if len(available) < 2:
        print(f"{exp['name']} overgeslagen: te weinig geldige features in dataset.")
        continue

    metrics = run_quality_experiment(df, target_col, available, exp['params'])
    row = {
        'experiment': exp['name'],
        'features': ', '.join(available),
        **exp['params'],
        **metrics,
    }
    results.append(row)

results_df = pd.DataFrame(results)
results_df

,experiment,features,max_depth,min_samples_leaf,accuracy,precision_weighted,recall_weighted,f1_weighted,min_samples_split
0,Exp 1,"Neighborhood, Gr Liv Area, Year Built",6,8,0.506826,0.511509,0.506826,0.483900,NaN
1,Exp 2,"Neighborhood, Gr Liv Area, Year Built, Total B...",10,4,0.520478,0.512377,0.520478,0.510649,12.0
2,Exp 3,"Neighborhood, Gr Liv Area, Year Built, Lot Are...",12,3,0.506826,0.501847,0.506826,0.501507,8.0
3,Exp 4,"Neighborhood, House Style, Gr Liv Area, Year B...",14,2,0.517065,0.513059,0.517065,0.507335,6.0


In [20]:
if results_df.empty:
    raise ValueError('Geen geldige experimenten uitgevoerd.')

print('Overzicht metrics:')
print(results_df[['experiment', 'accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']])

best_idx = results_df['f1_weighted'].idxmax()
print('\nBeste experiment op F1 (weighted):')
print(results_df.loc[best_idx])

Overzicht metrics:
  experiment  accuracy  precision_weighted  recall_weighted  f1_weighted
0      Exp 1  0.506826            0.511509         0.506826     0.483900
1      Exp 2  0.520478            0.512377         0.520478     0.510649
2      Exp 3  0.506826            0.501847         0.506826     0.501507
3      Exp 4  0.517065            0.513059         0.517065     0.507335

Beste experiment op F1 (weighted):
experiment                                                        Exp 2
features              Neighborhood, Gr Liv Area, Year Built, Total B...
max_depth                                                            10
min_samples_leaf                                                      4
accuracy                                                       0.520478
precision_weighted                                             0.512377
recall_weighted                                                0.520478
f1_weighted                                                    0.510649
min_

In [ ]:
baseline = results_df.iloc[0]
compare_cols = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']

print('Verschil t.o.v. Exp 1:')
for i in range(1, len(results_df)):
    current = results_df.iloc[i]
    print(f"\n{current['experiment']}")
    for c in compare_cols:
        print(f"{c}: {current[c] - baseline[c]:+.4f}")

NameError: name 'df' is not defined

## Vergelijking
Onderstaande cel toont het verschil tussen experiment 2 en de initiële run.

In [ ]:
# Handige variabelen voor je logboek
if len(results_df) >= 4:
    metrics_1 = results_df.iloc[0][['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']].to_dict()
    metrics_2 = results_df.iloc[1][['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']].to_dict()
    metrics_3 = results_df.iloc[2][['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']].to_dict()
    metrics_4 = results_df.iloc[3][['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']].to_dict()

print('Klaar: 4 experimenten uitgevoerd voor kwaliteit-classificatie.')

Vergelijking (experiment 2 - initiële run):


NameError: name 'metrics_1' is not defined